# v4 repaired and loubna's edits: line following, gaps, and green-marker turns

Based on the team's v4 green-marker design and the earlier gap controller. No motor movement starts automatically; uncomment the driving call deliberately. Run one cell at a time on the Jetson, never Run All. Shut down previous camera kernels first. Resolve power alarms before driving.

Copy working calibration from your robot: black threshold, bands, speeds, gains, reverse setting, and gap timeout. The defaults here are trial settings, not your measured values. Green-triggered turns reverse one wheel regardless of the ordinary-curve ALLOW_REVERSE setting.

Corrections: valid green detector with area/shape filtering and return value; OpenCV 3/4 contour compatibility; actual stop/advance/turn sequence; turn timers start with the turn; departure and multi-frame reacquisition; latched timeout stops; marker cooldown; restored gap handling and camera freshness checks.

Limits: marker geometry is heuristic. There is no full intersection topology detector or confirmation that a marker is before an intersection. U-turn angle is not measured: minimum duration and line reacquisition need calibration and can select the wrong branch. White gaps use bounded open-loop motion. Software timeouts do not protect against a frozen process/driver. Keep a physical stop accessible. Bye me: Loubna

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)

In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")

In [ ]:
BLACK_THRESHOLD = 70  # Copy your calibrated threshold here.
BANDS = [("near", 0.72, 0.88), ("middle", 0.54, 0.70), ("far", 0.36, 0.52)]

# --- new: vertical-support line tracking and junction detection ---
LINE_WIDTH_PX = 18       # tape width in the 224px image on a straight piece. CALIBRATE (use measure_line()).
SUPPORT = 0.75           # column must be black for this fraction of band height to count as "the line".
                         # Must be above LINE_WIDTH_PX / band height (~36px), otherwise side stubs count too.
JUNCTION_EXTRA_PX = 22   # a row is "crossbar" if black width > LINE_WIDTH_PX + this
JUNCTION_MIN_ROWS = 4
JUNCTION_ALIGN = 0.20    # continuation above the crossbar must be this close (normalized) to the line x

TRACK = {}               # last line x per band, cleared at the start of each run

def band_slice(mask, name):
    h = mask.shape[0]
    for n, top, bottom in BANDS:
        if n == name:
            return mask[int(top * h):int(bottom * h), :]

def vertical_runs(strip):
    h, w = strip.shape
    cols = (strip > 0).sum(axis=0)
    good = cols >= SUPPORT * h
    runs, start = [], None
    for x in range(w + 1):
        on = x < w and good[x]
        if on and start is None:
            start = x
        elif not on and start is not None:
            if x - start >= 3:
                runs.append((start, x - 1))
            start = None
    return runs, cols

def band_line_x(strip, prev):
    """x (pixels) of the vertical line nearest to prev, ignoring side stubs. None if no vertical line."""
    runs, cols = vertical_runs(strip)
    if not runs:
        return None
    run = min(runs, key=lambda r: abs((r[0] + r[1]) / 2.0 - prev))
    xs = np.arange(run[0], run[1] + 1)
    wts = cols[run[0]:run[1] + 1].astype(float)
    return float((xs * wts).sum() / wts.sum())

def crossbar_rows(mask, name):
    strip = band_slice(mask, name)
    return int(((strip > 0).sum(axis=1) > LINE_WIDTH_PX + JUNCTION_EXTRA_PX).sum())

def detect_junction(mask):
    """Returns (junction_start, crossbar_present).
    junction_start: crossbar in a band AND the vertical line continues in the band above it.
    crossbar_present: any band shows a crossbar (used to know when it has been passed)."""
    width = mask.shape[1]
    present = any(crossbar_rows(mask, n) >= JUNCTION_MIN_ROWS for n in ("near", "middle", "far"))
    start = False
    for lower, upper in (("middle", "far"), ("near", "middle")):
        if crossbar_rows(mask, lower) < JUNCTION_MIN_ROWS:
            continue
        xl = band_line_x(band_slice(mask, lower), width / 2.0)
        if xl is None:
            continue
        xu = band_line_x(band_slice(mask, upper), xl)
        if xu is not None and abs(xl - xu) / (width / 2.0) <= JUNCTION_ALIGN:
            start = True
    return start, present

def detect_curve(frame):
    height, width = frame.shape[:2]
    gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
    overlay = frame.copy()
    positions = {}
    for name, top, bottom in BANDS:
        y0, y1 = int(top * height), int(bottom * height)
        strip = mask[y0:y1, :]
        positions[name] = None
        cx, cy = band_line_x(strip, TRACK.get(name, width / 2.0)), (y0 + y1) // 2
        if cx is None:
            # Curve fallback: old largest-contour centroid (no vertical support on a sharp bend).
            contours = cv2.findContours(strip, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
            if contours:
                contour = max(contours, key=cv2.contourArea)
                fraction = cv2.contourArea(contour) / float(strip.size)
                moments = cv2.moments(contour)
                if 0.005 < fraction < 0.80 and moments["m00"] > 0:
                    cx = moments["m10"] / moments["m00"]
                    cy = int(moments["m01"] / moments["m00"]) + y0
        if cx is not None:
            positions[name] = (cx - width / 2.0) / (width / 2.0)
            TRACK[name] = cx
            cv2.circle(overlay, (int(cx), cy), 4, (0, 255, 0), -1)
        else:
            TRACK.pop(name, None)
        cv2.rectangle(overlay, (0, y0), (width - 1, y1), (255, 0, 0), 1)
        cv2.putText(overlay, name, (3, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 0, 255), 1)
    cv2.line(overlay, (width // 2, 0), (width // 2, height - 1), (0, 255, 255), 1)
    return positions, overlay, mask

def measure_line():
    """Run with the robot sitting on a straight line, then on a junction, to calibrate."""
    frame = latest["sample"][0]
    _, _, mask = detect_curve(frame)
    for name, _, _ in BANDS:
        runs, _ = vertical_runs(band_slice(mask, name))
        print(name, "run widths:", [b - a + 1 for a, b in runs], "crossbar rows:", crossbar_rows(mask, name))

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.65  # Require a nearby marker before initiating a turn.
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15

def detect_green(frame):
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in ('far', 'middle', 'near') if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height):
    if not markers:
        return 'NONE'
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return 'UNKNOWN'
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return 'UNKNOWN'
    dx, dy = dx / length, dy / length
    sides = {'LEFT': [], 'RIGHT': []}
    for marker in markers:
        if marker['y'] < height * GREEN_TRIGGER_Y:
            continue
        score = (marker['x'] - near_x) * dy - (marker['y'] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if score > 0 else 'RIGHT'].append(marker)
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'


## 3. Existing curve controller
Use measured reliable motor commands. Default curve speed is lower than base speed; raise it if a wheel stalls.


In [ ]:
# Copy your measured, reliable motor settings. These are initial guesses.
BASE_SPEED = 0.15
CURVE_SPEED = 0.13  # Must still reliably move the robot on the floor.
KP = 0.14
MAX_SPEED = 0.25
TURN_SPEED = 0.15
ALLOW_REVERSE = False  # True enables an in-place turn with one wheel reversing.
TURN_ENTER = 0.55
TURN_EXIT = 0.22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = 0.35
FRAME_TIMEOUT = 0.5

def clamp(value, low, high):
    return max(low, min(high, value))

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.stopped = False

    def stop(self, reason):
        self.stopped = True
        return 0.0, 0.0, "STOP: " + reason

    def pivot(self, direction):
        speed = clamp(TURN_SPEED, 0.0, MAX_SPEED)
        inner = -speed if ALLOW_REVERSE else 0.0
        return (speed, inner) if direction > 0 else (inner, speed)

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        visible = [(name, value) for name, value in positions.items() if value is not None]

        if not visible:
            if self.lost_started is None:
                self.lost_started = now
            if now - self.lost_started >= LOST_TIMEOUT:
                return self.stop("line not reacquired")
            if not self.last_direction:
                return self.stop("line lost without a known turn direction")
            if self.turn_started is None:
                self.turn_started = now
                self.turn_direction = self.last_direction
            if now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")
            left, right = self.pivot(self.turn_direction)
            return left, right, "SEARCH"

        self.lost_started = None
        weights = {"near": 0.45, "middle": 0.35, "far": 0.20}
        error = sum(weights[name] * value for name, value in visible) / sum(weights[name] for name, _ in visible)
        farthest = positions.get("far")
        if farthest is None:
            farthest = positions.get("middle")
        # Anticipate a bend from its displacement relative to the near strip.
        if near is not None and farthest is not None:
            error += 0.40 * (farthest - near)
        error = clamp(error, -1.0, 1.0)
        if abs(error) > 0.15:
            self.last_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            # Require the near line and combined direction to agree before exiting.
            if near is not None and abs(near) < TURN_EXIT and abs(error) < TURN_EXIT:
                self.turn_started = None
                self.turn_direction = 0
            elif now - self.turn_started >= TURN_TIMEOUT:
                return self.stop("turn timed out")

        if self.turn_started is None and abs(error) >= TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error > 0 else -1

        if self.turn_started is not None:
            left, right = self.pivot(self.turn_direction)
            return left, right, "TURN"

        bend = min(abs(error) / TURN_ENTER, 1.0)
        base = BASE_SPEED + (CURVE_SPEED - BASE_SPEED) * bend
        correction = KP * error
        left = clamp(base + correction, 0.0, MAX_SPEED)
        right = clamp(base - correction, 0.0, MAX_SPEED)
        return left, right, "FOLLOW"


## 4. Gap controller
Copy your calibrated gap timeout and speed here.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. Green turn state machine
Three matching processed frames request STOP → ADVANCE → TURN → FOLLOW. The small advance moves toward the junction; calibrate GREEN_TRIGGER_Y and GREEN_ADVANCE_TIME together. Set advance time to zero if already correctly positioned. Minimum turn duration begins after advance and is not a measured angle. A timeout stops, and the same marker is ignored until absent for several frames after cooldown.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v4(seconds=10, drive=False):
    robot.stop()
    TRACK.clear()
    controller = GreenNavigator()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, black_mask = detect_curve(frame)
            junction_start, crossbar = detect_junction(black_mask)
            markers, green_mask = detect_green(frame)
            height, width = frame.shape[:2]
            direction = classify_green_direction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, now, junction_start, crossbar)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                points = get_line_points(positions, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in markers:
                    x, y, w, h = marker['box']
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(black_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | junction=%s/%s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW', state, direction,
                    junction_start, crossbar, left, right)
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):
    run_v4(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v4(seconds=seconds, drive=True)

## 6. Runner
Preview computes commands but only stops motors. Every run creates fresh controller state. Camera frames are checked for freshness even when no new frame arrives.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v4(seconds=10, drive=False):
    robot.stop()
    TRACK.clear()
    controller = GreenNavigator()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, black_mask = detect_curve(frame)
            junction_start, crossbar = detect_junction(black_mask)
            markers, green_mask = detect_green(frame)
            height, width = frame.shape[:2]
            direction = classify_green_direction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, now, junction_start, crossbar)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                points = get_line_points(positions, width, height)
                for a, b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int, a)), tuple(map(int, b)), (255, 0, 255), 2)
                for marker in markers:
                    x, y, w, h = marker['box']
                    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(black_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | junction=%s/%s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW', state, direction,
                    junction_start, crossbar, left, right)
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):
    run_v4(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v4(seconds=seconds, drive=True)